# Enhanced Fertilizer Prediction with Ensemble Modeling

This notebook aims to predict the optimal fertilizer type based on soil and crop conditions using an ensemble of classifiers. 

The workflow includes:
1. **Data Loading and Initial Exploration (with Sampling)**: Load data and use a smaller sample for faster iteration.
2. **Exploratory Data Analysis (EDA)**: Detailed analysis of features and their relationship with the target variable.
3. **Data Preprocessing**: Handling categorical features and encoding the target variable.
4. **Feature Engineering**: Creating new features from existing ones to potentially improve model performance.
5. **Train-Test Split**: Dividing the data into training and testing sets.
6. **Handling Class Imbalance (SMOTE)**: Addressing potential imbalances in the target variable distribution.
7. **Train Multiple Classifiers**: Training Logistic Regression, Random Forest, and XGBoost classifiers.
8. **Model Evaluation**: Assessing individual model performance (optional) and preparing for ensemble.
9. **Save Models and Artifacts**: Saving the trained models and necessary mappings for future use.
10. **Create Reusable Ensemble Prediction Class**: Developing a class for easy prediction using an ensemble of the trained models.
11. **Generate Submission File**: Using the ensemble predictor to generate predictions for the test set.

## 1. Setup and Data Loading

Import necessary libraries and load the dataset. The dataset will be sampled to 100 rows for faster execution during development and testing.

In [ ]:
%pip install xgboost
%pip install imblearn
%pip install sklearn
%pip install scipy
%pip uninstall -y scikit-learn
%pip install scikit-learn==1.5.2
%pip install lightgbm
%pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import warnings
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from scipy.stats import mode # For hard voting in ensemble
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import os

# Global variable to set device for model training
USE_GPU = True # Set to False to use CPU

# Configure settings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

#### Original Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("irakozekelly/fertilizer-prediction")

print("Path to dataset files:", path)

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv')
    df = df.drop('id', axis=1)
    #test_df_cm = pd.read_csv('/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv')
except FileNotFoundError:
    print("Error: 'model/data.csv' not found. Make sure the path is correct.")
    try:
        df = pd.read_csv('model/data.csv') # If CWD is repo root
    except FileNotFoundError:
        print("Error: Also failed to load 'model/data.csv' from repo root. Please check file location.")
        df = pd.DataFrame() # Create an empty dataframe to avoid further errors if file is truly missing

if not df.empty:
    print("Dataset loaded successfully.")
    print("First 5 rows of the dataset:")
    display(df.head())
    print("\nDataset Information:")
    df.info()
    print("\nDescriptive Statistics:")
    display(df.describe())
    print("\nMissing values:")
    display(df.isnull().sum())


    
    # Sample the DataFrame for faster testing as per requirements
    # print(f"\nOriginal DataFrame shape: {df.shape}")
    # df = df.sample(n=100, random_state=42)
    # df.reset_index(drop=True, inplace=True) # Optional: reset index if needed
    # print(f"Sampled DataFrame shape: {df.shape}")
    # print("Using a sampled DataFrame of 100 rows for development.")
    # print("\nFirst 5 rows of the sampled dataset:")
    # display(df.head())



else:
    print("Dataset is empty. Cannot proceed.")

The column 'Humidity ' has an extra space in its name. Let's rename it for consistency.

In [ ]:
if 'Humidity ' in df.columns:
    df.rename(columns={'Humidity ': 'Humidity'}, inplace=True)
    print("Renamed 'Humidity ' to 'Humidity'.")
    print("Updated columns:", df.columns.tolist())

## 2. Exploratory Data Analysis (EDA)

In this section, we'll explore the data to understand its characteristics and relationships between variables.

### 2.1 Target Variable Analysis: 'Fertilizer Name'

In [ ]:
if not df.empty:
    print("Distribution of Fertilizer Name:")
    fertilizer_counts = df['Fertilizer Name'].value_counts()
    print(fertilizer_counts)
    
    plt.figure(figsize=(12, 7))
    sns.barplot(x=fertilizer_counts.index, y=fertilizer_counts.values, palette='viridis')
    plt.title('Distribution of Fertilizer Name')
    plt.xlabel('Fertilizer Name')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("DataFrame is empty, skipping target variable analysis.")

**Observation:** The dataset is well-balanced across the different fertilizer types. "14-35-14" is the most frequent class, while "Urea" and "DAP" are slightly less frequent, but there is no major class imbalance. This is a good foundation for building a classification model. Applying SMOTE, as planned, is a good practice to prevent any potential bias from the minor differences in class distribution.

### 2.2 Univariate Analysis: Numerical Features

Let's examine the distribution of each numerical feature.

In [ ]:
if not df.empty:
    numerical_features = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
    
    # Histograms
    df[numerical_features].hist(bins=20, figsize=(15, 10), layout=(2, 3), color='skyblue')
    plt.suptitle('Histograms of Numerical Features', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    # Box plots
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(numerical_features):
        plt.subplot(2, 3, i + 1)
        sns.boxplot(y=df[col], color='lightcoral')
        plt.title(col)
    plt.suptitle('Box Plots of Numerical Features', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
else:
    print("DataFrame is empty, skipping numerical feature analysis.")

### 2.3 Univariate Analysis: Categorical Features

Let's examine the distribution of 'Soil Type' and 'Crop Type'.

In [ ]:
if not df.empty:
    categorical_features = ['Soil Type', 'Crop Type']
    
    plt.figure(figsize=(18, 6))
    for i, col in enumerate(categorical_features):
        plt.subplot(1, 2, i + 1)
        sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='pastel')
        plt.title(f'Distribution of {col}')
        plt.xlabel('Count')
        plt.ylabel(col)
    plt.tight_layout()
    plt.show()
else:
    print("DataFrame is empty, skipping categorical feature analysis.")

**Observation:**
* The numerical features ('Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous') have distributions that are close to uniform, which is beneficial for the model as there are no significant outliers or skewness to handle.
* For categorical features, 'Soil Type' has 5 distinct categories, with 'Sandy' and 'Clayey' soils being the most common in the dataset.
* 'Crop Type' has a wider variety of categories, with 'Sugarcane', 'Paddy', 'Millets' and 'Barley' being the most frequently occurring crops.

### 2.4 Bivariate Analysis

Now, let's explore relationships between features and the target variable ('Fertilizer Name').

#### 2.4.1 Numerical Features vs. Target ('Fertilizer Name')

In [ ]:
if not df.empty:
    plt.figure(figsize=(20, 25))
    for i, col in enumerate(numerical_features):
        plt.subplot(len(numerical_features), 1, i + 1) # Adjusted layout for better readability
        sns.boxplot(data=df, x='Fertilizer Name', y=col, palette='Set3')
        plt.title(f'{col} by Fertilizer Name')
        plt.xticks(rotation=45, ha='right')
    plt.suptitle('Numerical Features vs. Fertilizer Name', fontsize=18, y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
else:
    print("DataFrame is empty, skipping numerical features vs. target analysis.")

#### 2.4.2 Categorical Features vs. Target ('Fertilizer Name')

In [ ]:
if not df.empty:
    for col in categorical_features:
        plt.figure(figsize=(14, 8))
        sns.countplot(data=df, y='Fertilizer Name', hue=col, palette='Spectral', order = df['Fertilizer Name'].value_counts().index)
        plt.title(f'Fertilizer Name Distribution by {col}')
        plt.xlabel('Count')
        plt.ylabel('Fertilizer Name')
        plt.legend(title=col, bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
else:
    print("DataFrame is empty, skipping categorical features vs. target analysis.")

**Observations:**
* **Nutrient Content:** The nutrient levels (Nitrogen, Potassium, Phosphorous) are strong predictors of the fertilizer type, as expected. For example, high Nitrogen values in the soil are strongly associated with the use of 'Urea' and 'DAP' fertilizers. This confirms the fundamental relationship between soil nutrient deficiency and fertilizer composition.
* **Environmental Factors:** Temperature and Humidity also show distinct patterns for different fertilizers. 'Urea' and '28-28' fertilizers are generally used in warmer and more humid conditions.
* **Categorical Interactions:** There are very strong associations between the type of crop, soil type, and the recommended fertilizer. For instance, the combination of 'Sugarcane' crop and 'Clayey' soil is predominantly linked with 'Urea' and '28-28' fertilizers, while 'Paddy' in 'Red' soil is often associated with 'DAP'. These interactions are crucial for the model's predictive power.

#### 2.4.3 Pair Plot of Numerical Features

In [ ]:
if not df.empty:
    # Using a subset of features for pairplot if it's too crowded, or all if manageable
    # For now, let's use all numerical features plus the target for hue
    plt.figure(figsize=(12,12)) # Ensure figure size is adequate
    pair_plot_df = df[numerical_features + ['Fertilizer Name']]
    sns.pairplot(pair_plot_df, hue='Fertilizer Name', palette='tab10') # tab10 is good for many categories
    plt.suptitle('Pair Plot of Numerical Features by Fertilizer Name', y=1.02)
    plt.show()
else:
    print("DataFrame is empty, skipping pair plot.")

**Observations:**
- The pair plot helps visualize how different fertilizer types cluster in the multi-dimensional space of numerical features.
- We can see clear separations for some fertilizers based on N, P, K values. For example, high N values distinctly mark 'Urea'.
- Relationships between features like Temperature and Humidity can also be observed, though their direct impact on fertilizer choice might be less distinct than N, P, K.

#### 2.4.4 Correlation Matrix of Numerical Features

In [ ]:
if not df.empty:
    correlation_matrix = df[numerical_features].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()
else:
    print("DataFrame is empty, skipping correlation matrix.")

**Observations:**
- **Potassium and Phosphorous** show a moderate positive correlation, suggesting they might sometimes be required together.
- **Humidity and Temperature** show a slight positive correlation.
- **Nitrogen** does not show strong correlations with other nutrients, indicating it might be an independent factor.
- Overall, multicollinearity among the original numerical features doesn't seem to be a major issue.

## 3. Data Preprocessing

This section covers the necessary preprocessing steps before feature engineering and model training.
- **Label Encoding Target Variable**: 'Fertilizer Name' will be converted to numerical labels.
- **One-Hot Encoding Categorical Features**: 'Soil Type' and 'Crop Type' will be converted into a numerical format using one-hot encoding.

### 3.1 Label Encoding Target Variable ('Fertilizer Name')

In [ ]:
if not df.empty:
    target_encoder = LabelEncoder()
    df['Fertilizer Name_Encoded'] = target_encoder.fit_transform(df['Fertilizer Name'])
    
    # Store the mapping for later use (e.g., decoding predictions)
    fertilizer_label_map = {label: i for i, label in enumerate(target_encoder.classes_)}
    fertilizer_index_to_name_map = {i: label for i, label in enumerate(target_encoder.classes_)}
    
    print("Fertilizer Name encoded.")
    print("Mapping of fertilizer names to encoded labels:")
    print(fertilizer_label_map)
    display(df[['Fertilizer Name', 'Fertilizer Name_Encoded']].head(10))
else:
    print("DataFrame is empty. Skipping target encoding.")

### 3.2 One-Hot Encoding Categorical Features ('Soil Type', 'Crop Type')

We will use pandas `get_dummies` for simplicity here. For a production pipeline, `sklearn.preprocessing.OneHotEncoder` within a `ColumnTransformer` is often preferred, especially to handle unseen categories in test data gracefully, but for this dataset structure, `get_dummies` is straightforward.

In [ ]:
if not df.empty:
    # Make a copy to keep the original df intact for reference if needed
    df_processed = df.copy()

    categorical_cols_to_encode = ['Soil Type', 'Crop Type']
    df_processed = pd.get_dummies(df_processed, columns=categorical_cols_to_encode, prefix=categorical_cols_to_encode, dtype=int)

    print("Categorical features one-hot encoded.")
    print("Shape of processed DataFrame:", df_processed.shape)
    print("First 5 rows of the processed DataFrame with one-hot encoded columns:")
    display(df_processed.head())
else:
    print("DataFrame is empty. Skipping one-hot encoding.")

The original 'Fertilizer Name', 'Soil Type', and 'Crop Type' columns (text versions) are no longer needed for modeling after encoding, so we can drop them from `df_processed`.

In [ ]:
if not df_processed.empty:
    # Drop original categorical columns and the original target text column
    columns_to_drop_after_encoding = ['Fertilizer Name'] # Soil Type and Crop Type were replaced by get_dummies
    # Check if these columns exist before dropping
    columns_to_drop_after_encoding = [col for col in columns_to_drop_after_encoding if col in df_processed.columns]
    
    if columns_to_drop_after_encoding:
        df_processed.drop(columns=columns_to_drop_after_encoding, inplace=True)
        print(f"Dropped columns: {columns_to_drop_after_encoding}")
        print("Current columns in df_processed:", df_processed.columns.tolist())
    else:
        print("No columns to drop or already dropped.")
    display(df_processed.head())
else:
    print("df_processed is empty. Skipping column drop.")

## 4. Feature Engineering

Now, let's create new features from the existing ones to potentially improve model performance. We'll create:
- Nutrient Ratios (N/P, N/K, P/K)
- Logarithmic transformations of numerical features
- Square transformations of numerical features

### 4.1 Nutrient Ratios
These ratios can capture the relative balance of nutrients, which can be more informative than absolute values alone.

In [ ]:
if not df_processed.empty:
    # Add a small epsilon to avoid division by zero or log(0)
    epsilon = 1e-6 
    
    # Ensure original nutrient columns exist
    if 'Nitrogen' in df_processed.columns and 'Phosphorous' in df_processed.columns:
        df_processed['N_P_ratio'] = df_processed['Nitrogen'] / (df_processed['Phosphorous'] + epsilon)
    else:
        print("Warning: Nitrogen or Phosphorous column not found for N_P_ratio.")

    if 'Nitrogen' in df_processed.columns and 'Potassium' in df_processed.columns:
        df_processed['N_K_ratio'] = df_processed['Nitrogen'] / (df_processed['Potassium'] + epsilon)
    else:
        print("Warning: Nitrogen or Potassium column not found for N_K_ratio.")
        
    if 'Phosphorous' in df_processed.columns and 'Potassium' in df_processed.columns:
        df_processed['P_K_ratio'] = df_processed['Phosphorous'] / (df_processed['Potassium'] + epsilon)
    else:
        print("Warning: Phosphorous or Potassium column not found for P_K_ratio.")
        
    print("Nutrient ratios created.")
    display(df_processed[['Nitrogen', 'Phosphorous', 'Potassium', 'N_P_ratio', 'N_K_ratio', 'P_K_ratio']].head())
else:
    print("df_processed is empty. Skipping nutrient ratio creation.")

### 4.2 Logarithmic Transformations
Log transformations can help normalize skewed distributions and handle wide ranges of values.

In [ ]:
if not df_processed.empty:
    
    # numerical_features = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
    
    for col in numerical_features: # numerical_features defined in EDA section
        if col in df_processed.columns:
    
            df_processed[f'log_{col}'] = np.log1p(df_processed[col])
        else:
            print(f"Warning: Column {col} not found for log transformation.")
            
    print("Logarithmic features created.")
    log_feature_cols = [f'log_{col}' for col in numerical_features if f'log_{col}' in df_processed.columns]
    if log_feature_cols:
        display(df_processed[log_feature_cols].head())
else:
    print("df_processed is empty. Skipping log transformations.")

### 4.3 Square Transformations
Square transformations can capture quadratic relationships.

In [ ]:
if not df_processed.empty:
    for col in numerical_features: # numerical_features defined in EDA section
        if col in df_processed.columns:
            df_processed[f'sq_{col}'] = df_processed[col]**2
        else:
            print(f"Warning: Column {col} not found for square transformation.")
            
    print("Square features created.")
    sq_feature_cols = [f'sq_{col}' for col in numerical_features if f'sq_{col}' in df_processed.columns]
    if sq_feature_cols:
        display(df_processed[sq_feature_cols].head())
else:
    print("df_processed is empty. Skipping square transformations.")

Let's check the final set of columns in our processed DataFrame.

### 4.5 Tempreture Transformations
Capture other potential relationships.

In [ ]:
df_processed['Temp_Hum_Interaction'] = df_processed['Temparature'] * df_processed['Humidity']
df_processed['Moisture_Temp_Interaction'] = df_processed['Moisture'] * df_processed['Temparature']


In [ ]:
df_processed.head(5)

In [ ]:
df_processed.describe

In [ ]:
if not df_processed.empty:
    print("Final columns in df_processed after feature engineering:")
    print(df_processed.columns.tolist())
    print("\nShape of df_processed:", df_processed.shape)
    display(df_processed.head())
else:
    print("df_processed is empty.")

## 5. Train-Test Split

Now we split the data into training and testing sets. The target variable is 'Fertilizer Name_Encoded'.

In [ ]:
if not df_processed.empty:
    X = df_processed.drop('Fertilizer Name_Encoded', axis=1)
    y = df_processed['Fertilizer Name_Encoded']
    
    # Storing feature names for later use (e.g. in the reusable class)
    feature_names = X.columns.tolist()
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("Data split into training and testing sets.")
    print("Shape of X_train:", X_train.shape)
    print("Shape of X_test:", X_test.shape)
    print("Shape of y_train:", y_train.shape)
    print("Shape of y_test:", y_test.shape)
    
    print("\nTraining target distribution:")
    print(y_train.value_counts(normalize=True).sort_index())
    print("\nTest target distribution:")
    print(y_test.value_counts(normalize=True).sort_index())
    
else:
    print("df_processed is empty. Skipping train-test split.")

The `stratify=y` argument ensures that the proportion of each class in the target variable is maintained in both the training and testing sets, which is important for classification tasks, especially with potentially imbalanced datasets (though ours is fairly balanced).

## 6. Handle Class Imbalance (SMOTE)

Although our EDA showed that the target classes are relatively balanced, applying SMOTE (Synthetic Minority Over-sampling Technique) can still be beneficial to ensure the model learns equally well from all classes, especially if some minor imbalances exist or to fulfill the requirement.

SMOTE will be applied **only to the training data** to prevent data leakage from the test set.

In [ ]:
if 'X_train' in locals() and 'y_train' in locals(): # Check if X_train, y_train exist
    print("Class distribution before SMOTE:")
    print(y_train.value_counts().sort_index())
    
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print("\nClass distribution after SMOTE:")
    print(y_train_smote.value_counts().sort_index())
    
    print("\nShape of X_train before SMOTE:", X_train.shape)
    print("Shape of X_train_smote after SMOTE:", X_train_smote.shape)
else:
    print("X_train and/or y_train are not defined. Skipping SMOTE application.")

## 7. Train Multiple Classifiers for Ensemble

We will now train a few different classifiers on the SMOTE-augmented training data (`X_train_smote`, `y_train_smote`). These models will then be used in an ensemble.

In [ ]:
# Define model directory
# Models will be saved in the root 'model' directory, relative to this notebook's location.
model_dir = 'model' 
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"Created directory: {model_dir}")
else:
    print(f"Model directory '{model_dir}' already exists or path is incorrect if run from a different location.")

# Ensure X_train_smote and y_train_smote are available before proceeding to model training cells
if 'X_train_smote' not in locals() or 'y_train_smote' not in locals():
    print("Error: X_train_smote and/or y_train_smote are not defined. Halting before model training.")
    # You might want to raise an error here or handle this case more robustly
    # For now, subsequent cells will check for these variables again.
else:
    print("Training data (X_train_smote, y_train_smote) found. Proceeding to model training sections.")

### 7.1 LightGBM Model

In [ ]:
if 'X_train_smote' in locals() and 'y_train_smote' in locals():
    print("Configuring LightGBM model...")
    lgbm_params = {
        'objective': 'multiclass',
        'metric': 'multi_logloss',
        'n_estimators': 1000,  # Simplified number of estimators
        'learning_rate': 0.01,
        'num_leaves': 31,
        'max_depth': -1, # No limit
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1
    }
    if USE_GPU:
        print("Configuring LightGBM for GPU.")
        lgbm_params['device'] = 'gpu'
        # Add any other GPU specific parameters if needed, e.g., gpu_platform_id, gpu_device_id
    else:
        print("Configuring LightGBM for CPU.")
        lgbm_params['device'] = 'cpu'
    
    lgbm_model = lgb.LGBMClassifier(**lgbm_params)
    print("LightGBM configured.")
else:
    print("Skipping LightGBM configuration as training data is not available.")

In [ ]:
if 'lgbm_model' in locals() and 'X_train_smote' in locals() and 'y_train_smote' in locals():
    print("Training LightGBM model...")
    lgbm_model.fit(X_train_smote, y_train_smote, eval_set=[(X_test, y_test)],callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=0)  # disables logging
    ])
    
    lgbm_model_path = os.path.join(model_dir, 'lightgbm_model.joblib')
    joblib.dump(lgbm_model, lgbm_model_path)
    print(f"LightGBM model trained and saved to: {lgbm_model_path}")

    # Evaluate LightGBM
    y_pred_lgbm = lgbm_model.predict(X_test)
    acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
    print(f"LightGBM Test Accuracy: {acc_lgbm:.4f}")
    
    # Confusion Matrix for LightGBM
    print("\nConfusion Matrix for LightGBM:")
    if 'fertilizer_index_to_name_map' in locals():
        class_names_lgbm = [fertilizer_index_to_name_map[i] for i in sorted(fertilizer_index_to_name_map.keys())]
    else:
        class_names_lgbm = sorted(y_test.unique()) # Fallback
        
    cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_lgbm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=class_names_lgbm, 
                yticklabels=class_names_lgbm)
    plt.title('LightGBM Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping LightGBM training as model or data is not available.")

### 7.2 CatBoost Model

In [ ]:
if 'X_train_smote' in locals() and 'y_train_smote' in locals():
    print("Configuring CatBoost model...")
    catboost_params = {
        'iterations': 1000,  # Simplified number of iterations
        'learning_rate': 0.01,
        'depth': 6, # Max depth of trees
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass',
        'random_seed': 42,
        'verbose': 0, # Suppress output during training
        'early_stopping_rounds': 20
    }
    if USE_GPU:
        print("Configuring CatBoost for GPU.")
        catboost_params['task_type'] = 'GPU'
        # catboost_params['devices'] = '0' # Optional: specify GPU device if multiple are available
    else:
        print("Configuring CatBoost for CPU.")
        catboost_params['task_type'] = 'CPU'

    catboost_model = CatBoostClassifier(**catboost_params)
    print("CatBoost configured.")
else:
    print("Skipping CatBoost configuration as training data is not available.")

In [ ]:
if 'catboost_model' in locals() and 'X_train_smote' in locals() and 'y_train_smote' in locals():
    print("Training CatBoost model...")
    catboost_model.fit(X_train_smote, y_train_smote, eval_set=[(X_test, y_test)], verbose=0)
    
    catboost_model_path = os.path.join(model_dir, 'catboost_model.joblib')
    joblib.dump(catboost_model, catboost_model_path)
    print(f"CatBoost model trained and saved to: {catboost_model_path}")

    # Evaluate CatBoost
    y_pred_catboost = catboost_model.predict(X_test)
    acc_catboost = accuracy_score(y_test, y_pred_catboost)
    print(f"CatBoost Test Accuracy: {acc_catboost:.4f}")
    
    # Confusion Matrix for CatBoost
    print("\nConfusion Matrix for CatBoost:")
    if 'fertilizer_index_to_name_map' in locals():
        class_names_catboost = [fertilizer_index_to_name_map[i] for i in sorted(fertilizer_index_to_name_map.keys())]
    else:
        class_names_catboost = sorted(y_test.unique()) # Fallback
        
    cm_catboost = confusion_matrix(y_test, y_pred_catboost)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_catboost, annot=True, fmt='d', cmap='Purples', 
                xticklabels=class_names_catboost, 
                yticklabels=class_names_catboost)
    plt.title('CatBoost Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping CatBoost training as model or data is not available.")

As seen above, SMOTE has balanced the number of samples for each class in the training set by generating synthetic samples for the minority classes.

### 7.3 XGBoost Model (for Ensemble)

We will now train the XGBoost classifier, which will also be part of our ensemble. It's trained on the SMOTE-augmented training data (`X_train_smote`, `y_train_smote`).

In [ ]:
if 'X_train_smote' in locals() and 'y_train_smote' in locals():
    # Determine the number of unique classes for XGBoost's num_class parameter
    num_classes = y_train_smote.nunique()

    early_stopping_callback = xgb.callback.EarlyStopping(
        rounds=50,
        save_best=True,  # Saves the best model found during early stopping
        data_name='validation_0',
    )

    # Define a fixed set of best-known or trial parameters
    xgb_params = {
        'objective': 'multi:softmax',
        'num_class': num_classes,
        'eval_metric': 'mlogloss',
        'use_label_encoder': False,
        'n_estimators': 1000, # Simplified to match others for sample data
        'max_depth': 7,
        'learning_rate': 0.01, # Adjusted to be common
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 1,
        'callbacks': [early_stopping_callback],
        'verbosity': 0,
        'random_state': 42
    }
    if USE_GPU:
        print("Configuring XGBoost for GPU.")
        xgb_params['tree_method'] = 'gpu_hist'
    else:
        print("Configuring XGBoost for CPU.")
        xgb_params['tree_method'] = 'hist' # or 'auto'
        
    xgb_classifier = xgb.XGBClassifier(**xgb_params)

    print("Training XGBoost classifier with early stopping...")
    xgb_classifier.fit(
        X_train_smote,
        y_train_smote,
        eval_set=[(X_test, y_test)],
        verbose=0  # Suppresses per-iteration logs
    )

    print("\nTraining complete.")
    print(f"Best model has {xgb_classifier.get_num_boosting_rounds()} boosting rounds.")

    # Save the trained XGBoost model for the ensemble
    xgb_model_path = os.path.join(model_dir, 'xgboost_ensemble_model.joblib') # model_dir defined in the previous cell
    joblib.dump(xgb_classifier, xgb_model_path)
    print(f"XGBoost model (for ensemble) trained and saved to: {xgb_model_path}")

    # Optionally evaluate here
    y_pred_xgb_train_cell = xgb_classifier.predict(X_test)
    acc_xgb_train_cell = accuracy_score(y_test, y_pred_xgb_train_cell)
    print(f"XGBoost Test Accuracy (from training cell): {acc_xgb_train_cell:.4f}")

    # Confusion Matrix for XGBoost (from training cell)
    print("\nConfusion Matrix for XGBoost")
    if 'fertilizer_index_to_name_map' in locals():
        class_names_xgb = [fertilizer_index_to_name_map[i] for i in sorted(fertilizer_index_to_name_map.keys())]
    else:
        class_names_xgb = sorted(y_test.unique()) # Fallback
        
    cm_xgb_train_cell = confusion_matrix(y_test, y_pred_xgb_train_cell)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_xgb_train_cell, annot=True, fmt='d', cmap='Oranges', 
                xticklabels=class_names_xgb, 
                yticklabels=class_names_xgb)
    plt.title('XGBoost Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("X_train_smote and/or y_train_smote are not defined. Skipping model training.")

**Evaluation Observations:**
- **Accuracy:** The overall accuracy of the model on the test set will be noted here.
- **Classification Report:** 
    - *Precision*: Indicates the proportion of positive identifications that were actually correct. 
    - *Recall*: Indicates the proportion of actual positives that were correctly identified.
    - *F1-score*: A weighted average of precision and recall. 
    We will look at these scores for each fertilizer type to understand where the model performs well and where it might struggle.
- **Confusion Matrix:** This provides a visual representation of model performance, showing correct and incorrect predictions for each class. The diagonal elements represent the number of points for which the predicted label is equal to the true label, while off-diagonal elements are those that are mislabeled by the classifier.

## 9. Save Model and Artifacts

We will save the trained XGBoost model, the label encoder mapping (fertilizer name to index), and the list of feature names used during training. These are essential for making predictions on new data with the reusable class.

In [ ]:
import os

# Define paths for saving artifacts
model_dir = 'model' # Path to the root 'model' directory
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"Created directory: {model_dir}")

label_map_path = os.path.join(model_dir, 'label_map.joblib')
feature_names_path = os.path.join(model_dir, 'feature_names.joblib') # Or .json

try:
    # Save the fertilizer label map (index_to_name_map for easy decoding)
    if 'fertilizer_index_to_name_map' in locals():
        joblib.dump(fertilizer_index_to_name_map, label_map_path)
        print(f"Fertilizer label map saved to: {label_map_path}")
    else:
        print("Fertilizer label map ('fertilizer_index_to_name_map') not found. Cannot save.")
        
    # Save the list of feature names
    if 'feature_names' in locals():
        joblib.dump(feature_names, feature_names_path)
        print(f"Feature names saved to: {feature_names_path}")
    else:
        print("Feature names list ('feature_names') not found. Cannot save.")
        
except Exception as e:
    print(f"Error saving artifacts: {e}")

The model, label map, and feature names are now saved and can be loaded by the prediction class.

## 10. Create Reusable Ensemble Prediction Class

Finally, we'll create a reusable Python class for making predictions using an ensemble of the trained models. This class will encapsulate all the necessary steps: loading the models and artifacts, preprocessing new input data, performing feature engineering, applying an ensemble strategy, and returning the predicted fertilizer name(s).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy.stats import mode

class FertilizerPredictor:
    def __init__(self, model_dir='model', # Adjusted default model_dir
                 model_filenames=['lightgbm_model.joblib', 
                                  'catboost_model.joblib', 
                                  'xgboost_ensemble_model.joblib']):
        
        self.model_paths = [os.path.join(model_dir, fname) for fname in model_filenames]
        # Artifacts like label_map and feature_names are assumed to be in the same model_dir
        self.label_map_path = os.path.join(model_dir, 'label_map.joblib') 
        self.feature_names_path = os.path.join(model_dir, 'feature_names.joblib')
        
        self.models = []
        self.original_numerical_features = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
        self.original_categorical_features = ['Soil Type', 'Crop Type']
        
        self._load_artifacts()

    def _load_artifacts(self):
        try:
            for model_path in self.model_paths:
                self.models.append(joblib.load(model_path))
            self.index_to_name_map = joblib.load(self.label_map_path)
            self.trained_feature_names = joblib.load(self.feature_names_path)
            print(f"{len(self.models)} models and other artifacts loaded successfully.")
        except FileNotFoundError as e:
            print(f"Error loading artifacts: {e}. Ensure models and artifacts are in specified paths.")
            self.models = []
            self.index_to_name_map = None
            self.trained_feature_names = None
        except Exception as e:
            print(f"An unexpected error occurred while loading artifacts: {e}")
            self.models = []
            self.index_to_name_map = None
            self.trained_feature_names = None

    def _preprocess_input(self, input_data):
        if not isinstance(input_data, pd.DataFrame):
            input_df = pd.DataFrame([input_data])
        else:
            input_df = input_data.copy()
        
        if 'Humidity ' in input_df.columns:
            input_df.rename(columns={'Humidity ': 'Humidity'}, inplace=True)

        for col in self.original_categorical_features:
            if col in input_df.columns:
                input_df[col] = input_df[col].astype('category')
        
        input_df_encoded = pd.get_dummies(input_df, columns=self.original_categorical_features, prefix=self.original_categorical_features, dtype=int)
        return input_df_encoded

    def _engineer_features(self, input_df):
        df_eng = input_df.copy()
        epsilon = 1e-6
        if 'Nitrogen' in df_eng.columns and 'Phosphorous' in df_eng.columns:
            df_eng['N_P_ratio'] = df_eng['Nitrogen'] / (df_eng['Phosphorous'] + epsilon)
        if 'Nitrogen' in df_eng.columns and 'Potassium' in df_eng.columns:
            df_eng['N_K_ratio'] = df_eng['Nitrogen'] / (df_eng['Potassium'] + epsilon)
        if 'Phosphorous' in df_eng.columns and 'Potassium' in df_eng.columns:
            df_eng['P_K_ratio'] = df_eng['Phosphorous'] / (df_eng['Potassium'] + epsilon)
        if 'Temparature' in df_eng.columns and 'Humidity' in df_eng.columns:
            df_eng['Temp_Hum_Interaction'] = df_eng['Temparature'] * df_eng['Humidity']
        if 'Temparature' in df_eng.columns and 'Moisture' in df_eng.columns:
            df_eng['Moisture_Temp_Interaction'] = df_eng['Moisture'] * df_eng['Temparature']
        for col in self.original_numerical_features:
            if col in df_eng.columns:
                df_eng[f'log_{col}'] = np.log1p(df_eng[col])
                df_eng[f'sq_{col}'] = df_eng[col]**2
        return df_eng

    def _align_features(self, input_df_engineered):
        aligned_df = pd.DataFrame(columns=self.trained_feature_names)
        for col in self.trained_feature_names:
            if col in input_df_engineered.columns:
                aligned_df[col] = input_df_engineered[col]
            else:
                aligned_df[col] = 0 
        return aligned_df[self.trained_feature_names]

    def predict(self, input_data, strategy='hard_voting'):
        if not self.models or self.index_to_name_map is None or self.trained_feature_names is None:
            print("Predictor not initialized properly or no models loaded. Cannot predict.")
            return None
        
        processed_df = self._preprocess_input(input_data)
        engineered_df = self._engineer_features(processed_df)
        final_df = self._align_features(engineered_df)
        
        try:
            if strategy == 'hard_voting':
                # Collect predictions from all models
                all_predictions = np.array([model.predict(final_df) for model in self.models])
                # Perform majority voting
                ensemble_prediction_encoded, _ = mode(all_predictions, axis=0, keepdims=False)
            elif strategy == 'soft_voting':
                # Collect probability predictions
                all_proba_predictions = np.array([model.predict_proba(final_df) for model in self.models])
                # Average the probabilities
                avg_proba = np.mean(all_proba_predictions, axis=0)
                ensemble_prediction_encoded = np.argmax(avg_proba, axis=1)
            else:
                raise ValueError("Unsupported voting strategy. Choose 'hard_voting' or 'soft_voting'.")

            if not isinstance(ensemble_prediction_encoded, np.ndarray):
                 ensemble_prediction_encoded = np.array([ensemble_prediction_encoded])
            
            predicted_fertilizer_names = [self.index_to_name_map.get(pred_code, "Unknown") for pred_code in ensemble_prediction_encoded]
            
            return predicted_fertilizer_names[0] if len(predicted_fertilizer_names) == 1 and not isinstance(input_data, pd.DataFrame) else predicted_fertilizer_names
        except Exception as e:
            print(f"Error during ensemble prediction: {e}")
            print("Input data after processing and engineering leading to error:")
            print(final_df.head())
            print(f"Expected columns: {self.trained_feature_names}")
            return None
        
    def predict_proba(self, input_data):
        if not self.models or self.trained_feature_names is None:
            print("Predictor not initialized properly or no models loaded. Cannot predict probabilities.")
            return None
        
        processed_df = self._preprocess_input(input_data)
        engineered_df = self._engineer_features(processed_df)
        final_df = self._align_features(engineered_df)
        
        try:
            all_proba_predictions = []
            for model in self.models:
                if hasattr(model, 'predict_proba'):
                    all_proba_predictions.append(model.predict_proba(final_df))
                else:
                    # Fallback for models without predict_proba: one-hot encode predictions
                    # This is a simplification; proper handling might require calibration or different ensemble strategy
                    num_classes = len(self.index_to_name_map)
                    predictions = model.predict(final_df)
                    proba_like = np.zeros((final_df.shape[0], num_classes))
                    for i, pred_class in enumerate(predictions):
                        proba_like[i, pred_class] = 1.0
                    all_proba_predictions.append(proba_like)
            
            avg_proba = np.mean(np.array(all_proba_predictions), axis=0)
            return avg_proba
        except Exception as e:
            print(f"Error during ensemble probability prediction: {e}")
            return None
            
    def predict_top_n(self, input_data, n=3):
        probabilities = self.predict_proba(input_data)
        if probabilities is None:
            return None
        
        # Ensure probabilities is a 2D array
        if probabilities.ndim == 1:
            probabilities = probabilities.reshape(1, -1)
            
        top_n_indices = np.argsort(probabilities, axis=1)[:, -n:][:, ::-1]
        top_n_fertilizers = []
        for indices_row in top_n_indices:
            fertilizers_row = [self.index_to_name_map.get(idx, "Unknown") for idx in indices_row]
            top_n_fertilizers.append(fertilizers_row)
            
        if not isinstance(input_data, pd.DataFrame) or (isinstance(input_data, pd.DataFrame) and len(input_data) == 1):
            return top_n_fertilizers[0] if top_n_fertilizers else [] 
        return top_n_fertilizers


### Testing the `FertilizerPredictor` Class

Let's instantiate the class and test it with a sample input. The sample input should be a dictionary or a Pandas DataFrame with the raw feature names.

In [ ]:
# Test the FertilizerPredictor class
model_files = [
    'lightgbm_model.joblib',
    'catboost_model.joblib',
    'xgboost_ensemble_model.joblib'
]
predictor_test_instance = FertilizerPredictor(model_dir='model', model_filenames=model_files)

if predictor_test_instance.models: # Check if artifacts loaded successfully
    # Sample input 1 (dictionary)
    sample_input_1 = {
        'Temparature': 28, 
        'Humidity': 55, 
        'Moisture': 40,
        'Soil Type': 'Clayey',
        'Crop Type': 'Wheat',
        'Nitrogen': 50,
        'Potassium': 30,
        'Phosphorous': 60
    }
    
    print("\n--- Testing Single Prediction (Hard Voting) ---")
    predicted_fertilizer_1_hard = predictor_test_instance.predict(sample_input_1, strategy='hard_voting')
    print(f"Prediction for Sample Input 1 (Hard Voting): {predicted_fertilizer_1_hard}")

    print("\n--- Testing Single Prediction (Soft Voting) ---")
    predicted_fertilizer_1_soft = predictor_test_instance.predict(sample_input_1, strategy='soft_voting')
    print(f"Prediction for Sample Input 1 (Soft Voting): {predicted_fertilizer_1_soft}")

    # Sample input 2 (DataFrame for batch prediction)
    sample_input_2_df = pd.DataFrame([
        {
            'Temparature': 34, 'Humidity': 65, 'Moisture': 45, 
            'Soil Type': 'Red', 'Crop Type': 'Ground Nuts', 
            'Nitrogen': 20, 'Potassium': 10, 'Phosphorous': 30
        },
        {
            'Temparature': 25, 'Humidity': 60, 'Moisture': 50, 
            'Soil Type': 'Sandy', 'Crop Type': 'Barley', 
            'Nitrogen': 70, 'Potassium': 40, 'Phosphorous': 20
        }
    ])
    print("\n--- Testing Batch Prediction (Hard Voting) ---")
    predicted_fertilizers_2_hard = predictor_test_instance.predict(sample_input_2_df, strategy='hard_voting')
    print(f"Predictions for Sample Input 2 (Hard Voting): {predicted_fertilizers_2_hard}")

    print("\n--- Testing Batch Prediction (Soft Voting) ---")
    predicted_fertilizers_2_soft = predictor_test_instance.predict(sample_input_2_df, strategy='soft_voting')
    print(f"Predictions for Sample Input 2 (Soft Voting): {predicted_fertilizers_2_soft}")
    
    print("\n--- Testing Top-N Predictions ---")
    top_n_fertilizers_1 = predictor_test_instance.predict_top_n(sample_input_1, n=3)
    print(f"Top 3 predictions for Sample Input 1: {top_n_fertilizers_1}")

    top_n_fertilizers_2 = predictor_test_instance.predict_top_n(sample_input_2_df, n=3)
    print(f"Top 3 predictions for Sample Input 2 (DataFrame): {top_n_fertilizers_2}")

    # Test with a known entry from the original dataset (e.g., first row)
    if not df.empty:
        print("\n--- Testing with the first row of the original dataset (Hard Voting) ---")
        first_row_original_df = df.iloc[[0]] 
        
        # Prepare input by dropping target columns if they exist (they would if df is from train.csv)
        first_row_input = first_row_original_df.drop(columns=['Fertilizer Name', 'Fertilizer Name_Encoded'], errors='ignore')
        
        prediction_first_row = predictor_test_instance.predict(first_row_input, strategy='hard_voting')
        print(f"Input data for first row:\n{first_row_input.to_dict(orient='records')[0]}")
        print(f"Predicted Fertilizer for first row (Hard Voting): {prediction_first_row}")
        if 'Fertilizer Name' in first_row_original_df.columns:
             print(f"Actual Fertilizer for first row: {first_row_original_df['Fertilizer Name'].iloc[0]}")
else:
    print("Predictor could not be initialized or models not loaded. Skipping tests.")

## 11. Generate Submission File using Ensemble Predictions

In this section, we will:
1. Load the `test.csv` data.
2. Use the updated `FertilizerPredictor`'s `predict_top_n` method (which internally uses the ensemble) to get the top 3 fertilizer predictions for each entry.
3. Format these predictions into a `submission.csv` file.

### 11.1 Load Test Data

In [ ]:
test_data_path = '/kaggle/input/playground-series-s5e6/test.csv' 
try:
    test_df = pd.read_csv(test_data_path)
    print(f"Test data loaded successfully from {test_data_path}")
    display(test_df.head())
    print("Test data information:")
    test_df.info()
except FileNotFoundError:
    print(f"Error: {test_data_path} not found. Please ensure the dummy test file exists.")
    test_df = pd.DataFrame() 

### 11.2 Get Top 3 Predictions for Test Data

In [ ]:
# Instantiate the predictor for submission, ensuring it uses the ensemble models
model_files_for_submission = [
    'lightgbm_model.joblib',
    'catboost_model.joblib',
    'xgboost_ensemble_model.joblib'
]
submission_predictor = FertilizerPredictor(model_dir='model', model_filenames=model_files_for_submission)

if not test_df.empty and submission_predictor.models: # Check if predictor.models is not empty
    print("Generating top 3 predictions for the test data using ensemble...")
    
    test_features_df = test_df.drop(columns=['id'], errors='ignore') 
    
    if 'Humidity ' in test_features_df.columns:
        test_features_df.rename(columns={'Humidity ': 'Humidity'}, inplace=True)
        print("Renamed 'Humidity ' to 'Humidity' in test data.")
        
    top_3_predictions_list = submission_predictor.predict_top_n(test_features_df, n=3)
    
    if top_3_predictions_list:
        print(f"Generated {len(top_3_predictions_list)} top 3 predictions.")
        for i in range(min(5, len(top_3_predictions_list))):
            print(f"ID {test_df['id'].iloc[i] if 'id' in test_df.columns else i}: {top_3_predictions_list[i]}")
        
        submission_fertilizer_names = [' '.join(preds) for preds in top_3_predictions_list]
    else:
        print("Failed to generate top 3 predictions.")
        submission_fertilizer_names = []
else:
    print("Test data is empty or predictor is not initialized with models. Skipping top-3 prediction generation.")
    submission_fertilizer_names = []

### 11.3 Create and Save Submission File

In [ ]:
if not test_df.empty and submission_fertilizer_names and len(test_df) == len(submission_fertilizer_names):
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Fertilizer Name': submission_fertilizer_names
    })
    
    submission_file_path = 'submission.csv' 

    try:
        submission_df.to_csv(submission_file_path, index=False)
        print(f"\nSubmission file created successfully at: {submission_file_path}")
        display(submission_df.head())
    except Exception as e:
        print(f"Error saving submission file: {e}")
        
elif test_df.empty:
    print("Test data was not loaded. Cannot create submission file.")
elif not submission_fertilizer_names:
    print("Submission fertilizer names were not generated. Cannot create submission file.")
else:
    print("Mismatch between length of test data IDs and predictions. Cannot create submission file.")